## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Skala Pengukuran Data](images/img_01_measurement_scales.png)

```
+-------------------------------------------------------------------------+
|                        SKALA PENGUKURAN DATA                            |
+-------------------------------------------------------------------------+
|                                                                         |
|  [1] NOMINAL  : Kategorik murni tanpa tingkatan (Biner/Label)           |
|                 Contoh: Gender (L/P), Metode Bayar (COD/E-Wallet)       |
|                 Operasi : (=, !=), Modus, One-Hot Encoding              |
|                                                                         |
|  [2] ORDINAL  : Kategorik berjenjang/berperingkat                       |
|                 Contoh: Rating Kepuasan (1-5), Tingkat Pendidikan       |
|                 Operasi : (=, !=, >, <), Median, Ordinal Encoding       |
|                                                                         |
|  [3] INTERVAL : Numerik dengan jarak tetap, TANPA nol mutlak            |
|                 Contoh: Suhu (°C), Nilai Tes Standar                    |
|                 Operasi : (+, -), Mean, Standar Deviasi                 |
|                                                                         |
|  [4] RASIO    : Numerik dengan jarak tetap dan MEMILIKI nol mutlak      |
|                 Contoh: Pendapatan, Harga Produk, Jumlah Transaksi      |
|                 Operasi : (+, -, *, /), Semua Operasi Matematika        |
+-------------------------------------------------------------------------+
```


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Konfigurasi visualisasi
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("Library berhasil dimuat!")


## 📂 3. Memuat Dataset dan Inspeksi Struktur Awal


In [ ]:
# Memuat dataset e-commerce
df = pd.read_csv("../datasets/01_ecommerce_sales_eda.csv")

print("=== Dimensi Dataset ===")
print(f"Jumlah Baris: {df.shape[0]}, Jumlah Kolom: {df.shape[1]}\n")

# Menampilkan 5 baris pertama
display(df.head())


## 🔍 4. Identifikasi Tipe Data dan Skala Pengukuran


In [ ]:
# Menampilkan ringkasan tipe data dan kelengkapan nilai
info_df = pd.DataFrame({
    'Nama Kolom': df.columns,
    'Tipe Data': df.dtypes.values,
    'Jumlah Non-Null': df.notnull().sum().values,
    'Jumlah Unik': df.nunique().values,
    'Contoh Nilai': [df[col].iloc[0] for col in df.columns]
})

# Menambahkan klasifikasi skala pengukuran
skala_map = {
    'order_id': 'Identifier / Nominal',
    'customer_age': 'Rasio (Numerik Kontinu/Diskrit)',
    'gender': 'Nominal (Biner)',
    'category': 'Nominal (Multikategori)',
    'payment_method': 'Nominal (Multikategori)',
    'quantity': 'Rasio (Diskrit)',
    'price_per_unit_k': 'Rasio (Kontinu)',
    'discount_pct': 'Rasio (Persentase)',
    'total_amount_k': 'Rasio (Kontinu)',
    'rating_score': 'Ordinal (Skala 1-5)',
    'delivery_days': 'Rasio (Diskrit)'
}
info_df['Skala Pengukuran'] = info_df['Nama Kolom'].map(skala_map)

print("=== Ringkasan Skala Pengukuran Variabel ===")
display(info_df)


## 🛠️ 5. Preprocessing & Feature Encoding untuk Variabel Kategori


In [ ]:
# 1. Ordinal Encoding untuk Variabel Ordinal (rating_score sudah numerik berurutan 1-5)
print("Distribusi Frekuensi Rating Score (Ordinal):")
display(df['rating_score'].value_counts().sort_index().to_frame(name='Frekuensi'))

# 2. One-Hot Encoding untuk Variabel Nominal (category & payment_method)
df_encoded = pd.get_dummies(df, columns=['category', 'payment_method'], prefix=['cat', 'pay'], drop_first=True)

print("Dimensi setelah One-Hot Encoding:", df_encoded.shape)
print("Kolom hasil encoding:")
print([col for col in df_encoded.columns if col.startswith(('cat_', 'pay_'))])


## 📈 6. Visualisasi Proporsi Variabel Nominal & Ordinal


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Kategori Produk (Nominal)
sns.countplot(data=df, x='category', ax=axes[0], order=df['category'].value_counts().index, palette='viridis')
axes[0].set_title('Distribusi Pesanan Berdasarkan Kategori (Nominal)', fontweight='bold')
axes[0].set_xlabel('Kategori Produk')
axes[0].set_ylabel('Jumlah Pesanan')
axes[0].tick_params(axis='x', rotation=30)

# Plot Rating Kepuasan (Ordinal)
sns.countplot(data=df, x='rating_score', ax=axes[1], palette='crest')
axes[1].set_title('Distribusi Skor Rating Kepuasan (Ordinal)', fontweight='bold')
axes[1].set_xlabel('Rating (Skala 1 - 5)')
axes[1].set_ylabel('Jumlah Ulasan')

plt.tight_layout()
plt.show()


## 📝 Kesimpulan Analisis

### Q&A
* **Apakah tipe data kategorik bisa langsung dimasukkan ke algoritma komputasi?** Tidak. Variabel nominal harus diubah melalui One-Hot Encoding (`pd.get_dummies`), sedangkan variabel ordinal dapat dipetakan secara terurut (*Integer/Ordinal Encoding*).
* **Mengapa penting membedakan interval dan rasio?** Variabel rasio memiliki nilai nol mutlak yang memungkinkan operasi perkalian/pembagian (misal: harga 200k adalah 2x lipat dari 100k), sedangkan variabel interval tidak memiliki nol mutlak.

### Data Analysis Key Findings
* Dataset terdiri dari **150 transaksi** dengan 11 fitur heterogen (kombinasi nominal, ordinal, dan rasio).
* Distribusi kategori produk merata di 5 kategori utama (*Electronics, Fashion, Groceries, Books, Home & Living*).
* Rating kepuasan konsumen didominasi oleh skor **4 dan 5** (>65% dari total pesanan).

### Insights or Next Steps
* Fitur hasil One-Hot Encoding siap digunakan sebagai matriks fitur ($X$) dalam pemodelan prediktif lanjutan.
* Lakukan pemeriksaan ukuran pemusatan dan penyebaran pada Modul 02 untuk mendeteksi potensi data pencilan (*outlier*).
